# CS604 Colab launcher — Ana: which box finder finds the organism? (no training)

The first training-crop pilot (`colab_launcher_ana_train_crops.ipynb`) tied: A 0.8978, bigger crops 0.8950, aimed crops 0.8950. But its boxes were barely better than random placement (box crops 0.591 vs random crops 0.581 with the prototype head), so aimed crops were never fairly tested.

**Part 1 (about 25 min, no training):** `scripts/box_sweep.py` runs 7 box-finding variants (`src/boxes.py`) on 200 of the pilot's 1,000 classes (fit images only) and scores each by how often the pretrained prototype head recognises its box crops, against crops of the same size placed at random or in the centre. A box finder that finds the organism should beat random placement clearly.

| Variant | Score map |
|---|---|
| `ddt_final` | the first pilot's boxes (DDT co-localization, final tokens) |
| `ddt_final_masked_smooth` | same, without high-norm outlier tokens, smoothed 3×3 |
| `ddt_block9_smooth`, `ddt_block9_masked_smooth` | DDT on block 10 of 12 (more spatial detail) |
| `ddt_block6_smooth` | DDT on block 7 of 12 |
| `proto_final_smooth`, `proto_final_masked_smooth` | true-label prototype map (cosine with the class's mean feature) |

Also printed: how many tokens per layer have an outlier norm (over 2× the photo's median), a check on the "artifact token" explanation for the spotty maps.

**Round 2 (what Part 1 runs now, about 15 min):** in round 1 no variant beat random placement, and a centred crop of the same size beat them all. The prototype map lit up the organism but kept about half of each photo, so its boxes were loose. Round 2 keeps only each photo's top 30% or 20% of patches, ranks every variant against the **centred crop**, and keeps two round-1 variants as references. Round 1's results stay in `box_sweep.csv`; round 2 writes `box_sweep_round2.*`.

**Part 2 (optional, about 20 min):** if a variant clearly beats the centred crop (say +3 points or more, with boxes on at least half the photos), set `RUN_BOX2 = True` and `BOX2_FLAGS` to its flags (printed by the sweep) in the first cell, and **Run all** again. Part 1 is reused; Part 2 rebuilds the pilot boxes with that variant (`boxes_v2.pt`) and trains one new arm, `BOX2_aimed_crops_sweep_winner`, next to the existing A / S25 / BOX results.

Results go to `CS604_RUNS/ablation_ablation_train_crops/`. Runtime: **A100 GPU**. Use **Runtime → Run all**.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
METHOD = "lora_r24_336_no_extras"  # base config; only used to build prototypes_336.pt if it is missing
PROTO_PT = f"{DRIVE_OUT}/prototypes_336.pt"
SPEC = "configs/ablation_train_crops.yaml"
PILOT_DATA_DIR = "/content/pilot_data"
FEATURE_DIR = "/content/features_pilot"  # local disk
# Part 2 (optional): after reading the sweep, e.g. BOX2_FLAGS = "--score ddt --layer 9 --mask --smooth"
RUN_BOX2 = False
BOX2_FLAGS = ""
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 4b. Build class prototypes at 336 (once)

Same as Van's prototype head, but built from 336 features (built with `configs/{METHOD}.yaml`, so at its `img_size`). Run **once**; about 20–25 min on an A100. Takes a while (embeds train images). Skip if the file already exists on Drive.

In [ ]:
from pathlib import Path

if Path(PROTO_PT).is_file():
    print("Already have", PROTO_PT)
else:
    !python scripts/build_prototypes.py --config configs/{METHOD}.yaml --data_root {DATA_DIR} --output {PROTO_PT}

## 5. Part 1: box sweep, round 2 (no training)

Builds the same pilot subset as the ablations (1,000 classes, seed 0, 5 holdout photos per class), then runs round 2 on 200 of its classes. Reused if `box_sweep_round2.csv` exists.

In [ ]:
import subprocess
import sys
from pathlib import Path

OUT_DIR = Path(DRIVE_OUT) / f"ablation_{Path(SPEC).stem}"
PILOT_ROOT = f"{PILOT_DATA_DIR}/c1000_h5_s0"  # built by scripts/ablation.py from the spec's pilot block


def run(cmd, what):
    proc = subprocess.run([sys.executable, "-u", *cmd])
    if proc.returncode != 0:
        end_runtime(f"{what} failed (exit {proc.returncode}); see the output above")
        raise SystemExit(f"{what} failed")


def ablation(*only):
    cmd = ["scripts/ablation.py", "--spec", SPEC, "--data_root", DATA_DIR, "--output_dir", DRIVE_OUT,
           "--pilot_data_dir", PILOT_DATA_DIR, "--feature_dir", FEATURE_DIR]
    run(cmd + (["--only", *only] if only else []), "ablation")


import json

import pandas as pd
from IPython.display import Image, display

from scripts.pilot_sweep import build_pilot_root

build_pilot_root(DATA_DIR, PILOT_DATA_DIR, 1000, 5, 0)  # same root as configs/ablation_train_crops.yaml
SWEEP = "box_sweep_round2"  # round 1: box_sweep.csv (already on Drive)
if not (OUT_DIR / f"{SWEEP}.csv").is_file():
    run(["scripts/box_sweep.py", "--config", f"configs/{METHOD}.yaml", "--data_root", PILOT_ROOT,
         "--out_dir", str(OUT_DIR), "--round", "round2"], "box sweep")
sweep = pd.read_csv(OUT_DIR / f"{SWEEP}.csv").sort_values("lift_vs_centre_points", ascending=False)
display(sweep)
display(Image(filename=str(OUT_DIR / f"{SWEEP}_overlay.png")))


## 6. Part 2 (optional): rebuild the pilot boxes with the winner and train BOX2

Only when `RUN_BOX2 = True`. Uses all 40,000 fit photos (~10 min), then one pilot arm (~6 min) plus data and prototypes reused from Drive.

In [ ]:
if RUN_BOX2:
    assert BOX2_FLAGS.strip(), "set BOX2_FLAGS in the first cell to the winning variant's flags"
    if not (OUT_DIR / "boxes_v2.pt").is_file():
        run(["scripts/find_boxes.py", "--config", f"configs/{METHOD}.yaml", "--data_root", PILOT_ROOT,
             "--out_dir", str(OUT_DIR), "--out_name", "boxes_v2", *BOX2_FLAGS.split()], "box finding (v2)")
    print(json.dumps(json.loads((OUT_DIR / "boxes_v2_stats.json").read_text()), indent=2))
    display(Image(filename=str(OUT_DIR / "boxes_v2_overlay.png")))
    ablation("A_random_crops", "S25_bigger_crops", "BOX_aimed_crops",  # reused
             "BOX2_aimed_crops_sweep_winner")
    table = pd.read_csv(OUT_DIR / "ablation.csv")
    display(table[["arm", "final_val_acc", "delta_vs_A_points", "best_val_acc", "backbone_pct",
                   "training_min", "peak_vram_gb", "wall_min"]])
else:
    print("Part 2 skipped (RUN_BOX2 = False)")


## 7. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")